# LLM GAME - 세종대왕과 표준말 배틀하기 


---

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

**[프롬프팅의 기본 구성]**

1. Instruction (지시사항 = 역할, 목표, 제약사항)
2. Context (문맥)
3. Input Data/Example (입력/예시)
4. Ouput Indicator (출력지시)
5. [ReAct 기법]

**Reason & Action 기법**
: 현재 상황에 대한 통찰 이후 다음 행동에 대한 작성을 유도하는 기법

**고어 및 고전 용어 활용 가능한 오픈 API**
- 국립국어원(우리말샘 API):https://opendict.korean.go.kr/service/openApiInfo
- 국립국어원(표준국어대사전 API):https://stdict.korean.go.kr/openapi/openApiInfo.do
- 한국학중앙연구원(한국학 지식정보 API): 요청url http://kostma.aks.ac.kr/OpenAPI/request.aspx

## Openai API를 doc에 담기

In [9]:
import os
import requests
import xml.etree.ElementTree as ET # XML 처리를 위한 도구
from dotenv import load_dotenv
from langchain_core.documents import Document

load_dotenv()
api_key = os.getenv("KOREAN_DICT_KEY")

def dict_to_docs(query):
    # 1. API 호출 (XML 형식)
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": api_key,
        "q": query,
        "method": "exact"
    }
    
    #실제 오는 내용
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        try:
            # 2. XML 파싱
            root = ET.fromstring(response.text)
            docs = []
            
            # 우리말샘 XML의 구조: <channel> -> <item>들
            for item in root.findall('.//item'):
                # <word> 태그와 <definition> 태그 내용 추출
                word = item.find('word').text if item.find('word') is not None else "단어 없음"
                # 뜻풀이: <sense> 안의 <definition>에 들어있음
                definition_tag = item.find('.//definition')
                definition = definition_tag.text if definition_tag is not None else "뜻 없음"
                
                content = f"단어: {word}\n뜻: {definition}"
                
                # 3. LangChain Document 객체로 변환
                doc = Document(
                    page_content=content,
                    metadata={"source": "우리말샘 API", "word": word}
                )
                docs.append(doc)
            
            if not docs:
                print(f"'{query}'에 대한 검색 결과가 없습니다.")
            return docs
            
        except ET.ParseError as e:
            print(f"파싱 에러: {e}")
            return []
    else:
        print(f"API 호출 실패 (상태 코드: {response.status_code})")
        return []

# 실행 테스트
docs = dict_to_docs("시나브로")
if docs:
    for d in docs:
        print(f"--- 문서 ---\n{d.page_content}\n")

--- 문서 ---
단어: 시나브로
뜻: 모르는 사이에 조금씩 조금씩.

--- 문서 ---
단어: 시나브로
뜻: 유리 상자의 앨범. 2001년에 발매되었다.



## 세종대왕 GPT 생성하기

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 우리나라의 세종대왕입니다. 옛날 훈민정음 창시자로서 우리나라의 고어를 매우 잘 압니다. 문자의 규칙에 대해 엄격하기 때문에,'
     '현대의 줄임말이나 외래어 등 표준말을 벗어난 말들에 대해 반감을 갖습니다. 아름다운 우리나라 말을 해치는 일이기 때문입니다. '
     '당신에게 표준말, 고어 외의 단어를 쓰는 사람이 있다면 따끔하게 혼내십시오.'),
    ('human', '''
            제공된 context만을 참고하여 질문에 답변해 주세요.
            context에서 확인할 수 없는 내용은 추론하지 말고 모른다면 직접 찾아보라고 잔소리하세요.
            최종 응답에는 참조한 context에 대한 정보를 추가해 주세요.

            질문: {question}
     
            context: {context}
     
            <<<최종응답 형식>>>
            답변:
            참조문서:
                - [source] (site): [doc]): [doc_content]
     ''')
])

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model('openai:gpt-4.1-mini')

In [ ]:
from langchain_chroma.vectorstores import Chroma

vector_store = Chroma.from_documents(documents=docs, embedding=embedding_model)
vector_store